# Solution 08 — the offline proof

All three claims, as plain `assert`s, zero API keys. If any assertion fails, the
notebook fails loudly — which is exactly how it works under `pytest` in CI.

In [ ]:
# ── SETUP — run this first in every notebook ─────────────────────────────────
import os
import sys

import logfire
from dotenv import find_dotenv, load_dotenv

# 1) Environment: walk up from notebooks/ to the repo root and load .env
load_dotenv(find_dotenv())

# 2) Observability: no Logfire account yet? This still works — data only ships
#    when a token exists (`uv run logfire auth`, or a LOGFIRE_TOKEN in .env).
logfire.configure(send_to_logfire="if-token-present")
logfire.instrument_pydantic_ai()  # v2: instrumentation is explicit — Agent(instrument=True) is gone

MODEL = os.getenv("MODEL", "openai:gpt-6.1-sol")

print("kernel :", sys.executable)  # should end in learningdantic/.venv/bin/python
print("MODEL  :", MODEL)

In [ ]:
from pydantic_ai import Agent, capture_run_messages, models
from pydantic_ai.messages import ModelResponse, TextPart, ToolCallPart, ToolReturnPart
from pydantic_ai.models.function import FunctionModel


def scripted_model(messages, info):
    """Call add(2, 3) first; answer with text once the tool result is back."""
    tool_ran = any(
        isinstance(part, ToolReturnPart) for message in messages for part in message.parts
    )
    if tool_ran:
        return ModelResponse(parts=[TextPart(content="The sum is 5.")])
    return ModelResponse(parts=[ToolCallPart(tool_name="add", args={"a": 2, "b": 3})])


spy = {"add_calls": 0}
offline_agent = Agent(FunctionModel(scripted_model), instructions="arith")


@offline_agent.tool_plain
def add(a: int, b: int) -> int:
    """Add two integers."""
    spy["add_calls"] += 1  # ← claim 1: the tool really executes
    return a + b


# Claim 3 wrapper: test doubles are exempt from the kill switch.
with models.override_allow_model_requests(False), capture_run_messages() as messages:
    r = await offline_agent.run("what is 2+3?")

# Claim 1: the spy counter moved — our function ran.
assert spy["add_calls"] == 1, f"add ran {spy['add_calls']} times, expected 1"
print("1. add really executed:", spy)

# Claim 2: ToolCallPart arrives before the matching ToolReturnPart.
kinds = [type(part).__name__ for message in messages for part in message.parts]
call_index = kinds.index("ToolCallPart")
return_index = kinds.index("ToolReturnPart")
assert call_index < return_index, f"order wrong: {kinds}"
print("2. call before return:", kinds)

# And the run completed correctly under the kill switch (claim 3).
assert r.output == "The sum is 5.", r.output
print("3. ran fully offline under override_allow_model_requests(False):", repr(r.output))

**Notes**

- The spy counter is the simplest possible test double — for HTTP tools you'd inject a
  fake `httpx` client via **deps** (notebook 05's pattern) and assert it was used.
- `override_allow_model_requests(False)` wrapping your whole test suite is how you make
  "unbillable" a *guarantee* instead of a hope.